In [2]:
import os
from pathlib import Path

import boto3
from botocore.exceptions import ClientError


def upload_directory(local_dir: Path, bucket: str, prefix: str = ""):
    """Recursively upload a local directory to S3."""
    local_dir = local_dir.resolve()
    if not local_dir.is_dir():
        raise ValueError(f"{local_dir} is not a directory")

    session = boto3.Session()  # uses your default AWS credentials
    s3_client = session.client("s3")

    prefix = prefix.strip("/")

    for root, _, files in os.walk(local_dir):
        for filename in files:
            local_path = Path(root) / filename
            rel_path = local_path.relative_to(local_dir)

            if prefix:
                s3_key = f"{prefix}/{rel_path.as_posix()}"
            else:
                s3_key = rel_path.as_posix()

            print(f"Uploading {local_path} -> s3://{bucket}/{s3_key}")
            try:
                s3_client.upload_file(str(local_path), bucket, s3_key)
            except ClientError as e:
                print(f"Failed to upload {local_path}: {e}")

In [4]:
from pathlib import Path
import os

# Configure these for your environment
BUCKET_NAME = "ishiki-ml-datasets" 
S3_PREFIX = "AMI_samples"        

# Local folder to upload (relative to notebook's working directory)
# Since notebook is in ami/ folder, AMI_samples should be in the same directory
LOCAL_AMI_SAMPLES = Path("AMI_samples").resolve()

if not LOCAL_AMI_SAMPLES.exists():
    raise FileNotFoundError(f"Local AMI_samples folder not found at {LOCAL_AMI_SAMPLES}")

print(f"Uploading from: {LOCAL_AMI_SAMPLES}")
print(f"Uploading to: s3://{BUCKET_NAME}/{S3_PREFIX}")

# Perform upload
upload_directory(LOCAL_AMI_SAMPLES, BUCKET_NAME, S3_PREFIX)

print(f"\nFinished uploading {LOCAL_AMI_SAMPLES} to s3://{BUCKET_NAME}/{S3_PREFIX}")


Uploading from: /Users/manand/Documents/context_aware_modeling/ami/AMI_samples
Uploading to: s3://ishiki-ml-datasets/AMI_samples
Uploading /Users/manand/Documents/context_aware_modeling/ami/AMI_samples/.DS_Store -> s3://ishiki-ml-datasets/AMI_samples/.DS_Store
Uploading /Users/manand/Documents/context_aware_modeling/ami/AMI_samples/json_dumps/stage1_sequences.json -> s3://ishiki-ml-datasets/AMI_samples/json_dumps/stage1_sequences.json
Uploading /Users/manand/Documents/context_aware_modeling/ami/AMI_samples/json_dumps/stage3_categorized_samples.json -> s3://ishiki-ml-datasets/AMI_samples/json_dumps/stage3_categorized_samples.json
Uploading /Users/manand/Documents/context_aware_modeling/ami/AMI_samples/json_dumps/stage1b_inferred_sequences.json -> s3://ishiki-ml-datasets/AMI_samples/json_dumps/stage1b_inferred_sequences.json
Uploading /Users/manand/Documents/context_aware_modeling/ami/AMI_samples/json_dumps/stage2_decision_points.json -> s3://ishiki-ml-datasets/AMI_samples/json_dumps/sta